# I2C device initialization

In [ ]:
from time import sleep as delay
from project.sc8583 import project

if "ic" in globals():
    ic.close()
    
ic = project(device="sc8583", revision="1p1", emulator="ch341", logging=False, is_gui=False)

# Equipments initialization

In [ ]:
from phy.multimeter import keithley_dm6500
from phy.power_supply import rigol_dp821a, rigol_dp811a
from phy.power_analyzer import keysight_N6705
from phy.scope import tektronix_mdo34
from phy.battery_simulator import asd_906b

# from phy.eloader import sdl1030x
# from phy.relay_16ch import relay_box

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

import numpy as np
from time import sleep as delay
chart = plot()

dm = keithley_dm6500()
# ps1 = rigol_dp821a()
# ps2 = rigol_dp811a()
ps = keysight_N6705()
ds = tektronix_mdo34()
bs = asd_906b(port=7)
# sm = keithley_2470()
# ld = sdl1030x()

# relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)

# --------------------------------------------------
# list_voltage = list(np.arange(5, 8, 0.005))
# voltage  = [round(num, 3) for num in list_voltage]
# --------------------------------------------------

from concurrent.futures import ThreadPoolExecutor

# Test code block

In [ ]:
ic.update_i2c_address(0x6e)

In [ ]:
ds.ch1.vertical_scale = 5
ds.ch2.vertical_scale = 5
ds.ch3.vertical_scale = 1
ds.ch4.vertical_scale = 1

ds.ch1.vertical_grid = 0
ds.ch2.vertical_grid = -3
ds.ch3.vertical_grid = -2
ds.ch4.vertical_grid = -1

ds.ch1.label = "VIN"
ds.ch2.label = "C2NA"
ds.ch3.label = "VOUT"
ds.ch4.label = "IIN"

In [ ]:
def preset(op_mode):
    
    ic.MODE = op_mode
    ic.SS_TIMEOUT = 3
    ic.FREQ_SHIFT = 0
    ic.FSW_SET = 8
    ic.SET_IBAT_SNS_RES = 0
    ic.PMID_IN_RANGE_DIS = 0
    ic.VEXT_SHUTDOWN_SET = 0
    ic.STANDBY_MODE_SET = 1

    ic.VEXT_OVP = 12 # 23V
    ic.IIN_OCP = 0b0101_1010 # 3.375
    ic.IIN_REG = 0b0101_0110 # 3.225A

    # ic.VBAT_REG = 0b1101_0001 # 4.445
    ic.VBAT_REG = 0b1011_0100 # 4.3V

    ic.IIN_UCP_DIS = 1

    # ic.VBAT_OVP = 0b1001_1100 # 4.58
    ic.VBAT_OVP = 0b1011_1100 # 4.72
    
    ic.VIN_OVP = 0b1010 # 23V
    ic.WPC_IN_OVP = 0b1010 # 23V
    ic.IIN_OCP = 0b0110_0000 # 3.600A
    ic.VOUT_OVP = 2 # 5.1V
    ic.C1P2OUT_OVP = 3
    ic.C1P2OUT_UVP = 1

    ic.IIN_OCP_DG = 1 # 80us
    ic.VBAT_OVP_DG_SET = 0 # no deglitch
    ic.VOUT_OVP_DG_SET = 0 # no deglitch
    ic.VEXT_OVP_DG_SET = 0 # no deglitch
    ic.VIN_OVP_DG_SET = 0 # no deglitch
    ic.WPC_OVP_DG_SET = 0 # no deglitch
    ic.IBUS_UCP_FALL_DG_SET = 2 # 20ms

    ic.IBAT_REG_DIS = 1
    ic.IBAT_OCP_DIS = 1
    ic.NTC_FLT_DIS = 1
    ic.TDIE_REG_DIS = 1

    ic.ADC_EN = 1

In [ ]:
def reg_print():
    print(f"MODE = {ic.MODE}")
    print(f"SS_TIMEOUT = {ic.SS_TIMEOUT}")
    print(f"FREQ_SHIFT = {ic.FREQ_SHIFT}")
    print(f"FSW_SET = {ic.FSW_SET}")
    print(f"SET_IBAT_SNS_RES = {ic.SET_IBAT_SNS_RES}")
    print(f"PMID_IN_RANGE_DIS = {ic.PMID_IN_RANGE_DIS}")
    print(f"VEXT_SHUTDOWN_SET = {ic.VEXT_SHUTDOWN_SET}")
    print(f"STANDBY_MODE_SET = {ic.STANDBY_MODE_SET}")
    print(f"VEXT_OVP = {ic.VEXT_OVP}")
    print(f"IIN_OCP = {ic.IIN_OCP}")
    print(f"IIN_REG = {ic.IIN_REG}")
    print(f"VBAT_REG = {ic.VBAT_REG}")
    print(f"IIN_UCP_DIS = {ic.IIN_UCP_DIS}")
    print(f"VBAT_OVP = {ic.VBAT_OVP}")
    print(f"VIN_OVP = {ic.VIN_OVP}")
    print(f"WPC_IN_OVP = {ic.WPC_IN_OVP}")
    print(f"IIN_OCP = {ic.IIN_OCP}")
    print(f"VOUT_OVP = {ic.VOUT_OVP}")
    print(f"C1P2OUT_OVP = {ic.C1P2OUT_OVP}")
    print(f"C1P2OUT_UVP = {ic.C1P2OUT_UVP}")
    print(f"IIN_OCP_DG = {ic.IIN_OCP_DG}")
    print(f"VBAT_OVP_DG_SET = {ic.VBAT_OVP_DG_SET}")
    print(f"VOUT_OVP_DG_SET = {ic.VOUT_OVP_DG_SET}")
    print(f"VEXT_OVP_DG_SET = {ic.VEXT_OVP_DG_SET}")
    print(f"VIN_OVP_DG_SET = {ic.VIN_OVP_DG_SET}")
    print(f"WPC_OVP_DG_SET = {ic.WPC_OVP_DG_SET}")
    print(f"IBUS_UCP_FALL_DG_SET = {ic.IBUS_UCP_FALL_DG_SET}")
    print(f"IBAT_OCP_DIS = {ic.IBAT_OCP_DIS}")
    print(f"NTC_FLT_DIS = {ic.NTC_FLT_DIS}")
    print(f"TDIE_REG_DIS = {ic.TDIE_REG_DIS}")

In [ ]:
def trigger():
    ds.ch4.trigger_fall = 0.5
    ds.horizontal_grid = -2
    ds.single

# main abnormal test block

In [ ]:
# change the label if needed

vin_path = True

if vin_path:
    ds.ch1.label = "VIN"
else:
    ds.ch1.label = "WPC_IN"

bs.vset = 3.8

In [ ]:
# mode configuration
# 0 : FWD 4TO1
# 1 : FWD 3TO1
# 2 : FWD 2TO1
# 3 : FWD 1TO1

op_mode = 0

if op_mode == 0:
    ds.ch1.vertical_scale = 6
elif op_mode == 1:
    ds.ch1.vertical_scale = 4
elif op_mode == 2:
    ds.ch1.vertical_scale = 3
elif op_mode == 3:
    ds.ch1.vertical_scale = 2

In [ ]:
# configuration for suffix 1

ps.ch1.disable
bs.disable

single_shot = True
iteration_shot = False
count = 1

ds.horizontal_scale = 8e-6

delay(0.5)

ds.normal
delay(0.5)
ds.ch4.trigger_fall = 1
ds.ch4.vertical_scale = 1

suffix = "1"

delay(0.5)
bs.enable
ps.ch1.enable

delay(0.5)
ds.single

ds.horizontal_grid = 0

In [ ]:
# configuration for suffix 2

single_shot = True
iteration_shot = False
count = 1

ds.normal
ds.ch4.trigger_fall = 1
ds.horizontal_scale = 200e-3
ds.horizontal_grid = 2
ds.single

suffix = "2"

In [ ]:
# configuration for suffix 3

single_shot = False
iteration_shot = True
count = 5

ds.roll
ds.horizontal_scale = 2
ds.run

suffix = "3"

In [ ]:
target = 2

ds.ch1.vertical_grid = 0
ds.ch3.vertical_grid = -2
ds.ch4.vertical_grid = -1

if target == 4:
    vin_overshoot = 20
elif target == 3:
    vin_overshoot = 13
elif target == 2:
    vin_overshoot = 9
elif target == 1:
    vin_overshoot = 5
else:
    print("wrong vin overshoot")

In [ ]:
protection = "C1P2OUT_OVP"

def func_2():
    delay(0.04)
    ps.ch1.vset = vin_overshoot
    return "func #2 is done"

def func_1():
    ic.C1P2OUT_OVP = 0
    return "func #1 is done"

# ---------------------------

bs.vset = 3.8
delay(1)

ds.single

for i in range(count):

    ps.ch1.vset = 16
    preset(op_mode)
    if vin_path:
        ic.enable_vin_charging
    else:
        ic.enable_wpc_charging
    
    delay(0.1)

    ps.ch1.vset = 17.7

    if single_shot:
        delay(1)
    
    if iteration_shot:
        delay(2)

    with ThreadPoolExecutor(max_workers=2) as executor:
        thread_1 = executor.submit(func_1)
        thread_2 = executor.submit(func_2)
        ret_1 = thread_1.result()
        ret_2 = thread_2.result()

    delay(1)
    bs.vset = 3.8

In [ ]:
ds.save_waveform = f"{protection} - vin overshoot {vin_overshoot} #{suffix}"

In [ ]:
ic.status

# debugging

In [ ]:
ic.disable_charging

In [ ]:
alex = {
    0x00 : 0xa5,    0x01 : 0x00,    0x02 : 0x02,    0x03 : 0x85,    0x04 : 0x40,    0x05 : 0x00,    0x06 : 0x00,    0x07 : 0x00,    0x08 : 0x00,    0x09 : 0x00,    0x0a : 0x00,    0x0b : 0x00,    0x0c : 0x00,    0x0d : 0x3f,    0x0e : 0xa0,    0x0f : 0x00,    0x10 : 0x00,    0x11 : 0x00,    0x12 : 0x00,    0x13 : 0x00,    0x14 : 0x30,    0x15 : 0x08,    0x16 : 0x00,    0x17 : 0x23,    0x18 : 0x8c,    0x19 : 0x40,    0x1a : 0x23,    0x1b : 0xe6,    0x1c : 0x68,    0x1d : 0x19,    0x1e : 0xf0,    0x1f : 0xec,    0x20 : 0x50,    0x21 : 0x50,    0x22 : 0x2b,    0x23 : 0x40,    0x24 : 0x63,    0x25 : 0x01,    0x26 : 0xa0,    0x27 : 0x00,    0x28 : 0x10,    0x29 : 0x08,    0x2a : 0x40,    0x2b : 0x00,    0x2c : 0x02,    0x2d : 0x64,    0x2e : 0x0b,    0x2f : 0xb0,    0x30 : 0x00,    0x31 : 0x03,    0x32 : 0x0b,    0x33 : 0xb1,    0x34 : 0x0e,    0x35 : 0x5d,    0x36 : 0x0e,    0x37 : 0x33,    0x38 : 0x00,    0x39 : 0x32,    0x3a : 0x0b,    0x3b : 0xaf,    0x3c : 0x01,    0x3d : 0x6b,    0x3e : 0x00,    0x3f : 0x58}

In [ ]:
ic.log_analyzer(alex)

In [ ]:
0xe33 * ic.lsb_vout

In [ ]:
zech = {
    0x00 : 0xa5,    0x01 : 0x00,    0x02 : 0x00,    0x03 : 0x00,    0x04 : 0x00,    0x05 : 0x00,    0x06 : 0x00,    0x07 : 0x00,    0x08 : 0x00,    0x09 : 0x00,    0x0a : 0x00,    0x0b : 0x00,    0x0c : 0x00,    0x0d : 0x7f,    0x0e : 0xa2,    0x0f : 0x20,    0x10 : 0x00,    0x11 : 0x00,    0x12 : 0x00,    0x13 : 0xc0,    0x14 : 0x30,    0x15 : 0x08,    0x16 : 0x00,    0x17 : 0x23,    0x18 : 0x8c,    0x19 : 0x40,    0x1a : 0x23,    0x1b : 0xe6,    0x1c : 0x68,    0x1d : 0x19,    0x1e : 0xf0,    0x1f : 0xec,    0x20 : 0x50,    0x21 : 0x50,    0x22 : 0x2b,    0x23 : 0x40,    0x24 : 0x63,    0x25 : 0x01,    0x26 : 0xa0,    0x27 : 0x00,    0x28 : 0x10,    0x29 : 0x08,    0x2a : 0x40,    0x2b : 0x00,    0x2c : 0x00,    0x2d : 0x03,    0x2e : 0x0b,    0x2f : 0x51,    0x30 : 0x00,    0x31 : 0x02,    0x32 : 0x0b,    0x33 : 0x4e,    0x34 : 0x0d,    0x35 : 0xd5,    0x36 : 0x0d,    0x37 : 0xdb,    0x38 : 0x00,    0x39 : 0x33,    0x3a : 0x00,    0x3b : 0x03,    0x3c : 0x00,    0x3d : 0x13,    0x3e : 0x00,    0x3f : 0x4d}

In [ ]:
ic.log_analyzer(zech)

In [ ]:
zech2 = {0x00 : 0xa5, 0x01 : 0x00, 0x02 : 0x00, 0x03 : 0x80, 0x04 : 0x00, 0x05 : 0x00, 0x06 : 0x00, 0x07 : 0x00, 0x08 : 0x00, 0x09 : 0x00, 0x0a : 0x00, 0x0b : 0x00, 0x0c : 0x00, 0x0d : 0x3f, 0x0e : 0xa0, 0x0f : 0x00, 0x10 : 0x00, 0x11 : 0x00, 0x12 : 0x00, 0x13 : 0x00, 0x14 : 0x30, 0x15 : 0x08, 0x16 : 0x00, 0x17 : 0x23, 0x18 : 0x8c, 0x19 : 0x40, 0x1a : 0x23, 0x1b : 0xe6, 0x1c : 0x68, 0x1d : 0x19, 0x1e : 0xf0, 0x1f : 0xec, 0x20 : 0x50, 0x21 : 0x50, 0x22 : 0x2b, 0x23 : 0x40, 0x24 : 0x63, 0x25 : 0x01, 0x26 : 0xa0, 0x27 : 0x00, 0x28 : 0x10, 0x29 : 0x08, 0x2a : 0x40, 0x2b : 0x00, 0x2c : 0x00, 0x2d : 0x02, 0x2e : 0x0b, 0x2f : 0xcd, 0x30 : 0x00, 0x31 : 0x03, 0x32 : 0x0b, 0x33 : 0xc9, 0x34 : 0x0d, 0x35 : 0xce, 0x36 : 0x0d, 0x37 : 0xda, 0x38 : 0x00, 0x39 : 0x33, 0x3a : 0x00, 0x3b : 0x03, 0x3c : 0x00, 0x3d : 0xae, 0x3e : 0x00, 0x3f : 0x4e}

In [ ]:
ic.log_analyzer(zech2)

In [ ]:
0xdda * ic.lsb_vbat

In [ ]:
ic.status_ctrl

In [ ]:
bs.vset = 4.45
ps.ch1.vset = 18.5

In [ ]:
ic.WD_TIMEOUT_DIS = 1

In [ ]:
ic.enable_vin_charging

In [ ]:
ic.status

In [ ]:
def init():
    ic.write_byte(0x14, 0x30)
    ic.write_byte(0x15, 0x08)
    ic.write_byte(0x16, 0x00)
    ic.write_byte(0x17, 0x23)
    ic.write_byte(0x18, 0x8c)
    ic.write_byte(0x19, 0x40)
    ic.write_byte(0x1a, 0x23)
    ic.write_byte(0x1b, 0xe6)
    ic.write_byte(0x1c, 0x68)
    ic.write_byte(0x1d, 0x19)
    ic.write_byte(0x1e, 0xf0)
    ic.write_byte(0x1f, 0xec)
    ic.write_byte(0x20, 0x50)
    ic.write_byte(0x21, 0x50)
    ic.write_byte(0x22, 0x2b)
    ic.write_byte(0x23, 0x40)
    ic.write_byte(0x24, 0x63)
    ic.write_byte(0x25, 0x01)
    ic.write_byte(0x26, 0xa0)
    ic.write_byte(0x27, 0x00)
    ic.write_byte(0x28, 0x10)
    ic.write_byte(0x29, 0x08)
    ic.write_byte(0x2a, 0x40)

    ic.WD_TIMEOUT_DIS = 1

In [ ]:
init()

In [ ]:
print(f"CP_SWITCHING_STAT = {ic.CP_SWITCHING_STAT}")
print(f"VBAT_REG_ACTIVE_STAT = {ic.VBAT_REG_ACTIVE_STAT}")
print(f"IIN_UCP_FALL_FLAG = {ic.IIN_UCP_FALL_FLAG}")
print(f"IIN_REG_ACTIVE_STAT = {ic.IIN_REG_ACTIVE_STAT}")

In [ ]:
vin_start = 18.2
vout_start = 4.1

bs.vset = vout_start
ps.ch1.vset = vin_start
delay(0.5)

# bs.enable
# ps.ch1.enable
# delay(1)

init()

ic.enable_vin_charging
delay(1)

while True:
    
    sw_sts = ic.CP_SWITCHING_STAT
    vbat_reg_sts = ic.VBAT_REG_ACTIVE_STAT
    uc_flag = ic.IIN_UCP_FALL_FLAG
    iin_reg_sts = ic.IIN_REG_ACTIVE_STAT

    if sw_sts != 1:
        break

    if vin_start > 19.5:
        break
    
    if vout_start > 4.55:
        break

    vin_start += 0.015
    vout_start += 0.01

    bs.vset = vout_start
    ps.ch1.vset = vin_start

    iin = ps.ch1.current
    vbat = dm.voltage_10

    print(f"CP_SWITCHING_STAT={sw_sts} VBAT_REG_ACTIVE_STAT={vbat_reg_sts} IIN_UCP_FALL_FLAG={uc_flag} IIN_REG_ACTIVE_STAT={iin_reg_sts} VIN={vin_start:.03f} VBAT={vbat:.03f} IIN={iin:.03f}")

for _ in range(5):
    print(f"CP_SWITCHING_STAT={sw_sts} VBAT_REG_ACTIVE_STAT={vbat_reg_sts} IIN_UCP_FALL_FLAG={uc_flag} IIN_REG_ACTIVE_STAT={iin_reg_sts} VIN={vin_start:.03f} VBAT={vbat:.03f} IIN={iin:.03f}")

In [ ]:
ic.disable_charging

In [ ]:
ds.save_waveform = "SC8583 - PSG VBAT_OVP test"